<div style="border-top:4px solid #0f766e;padding:28px 0 18px"><div style="color:#0f766e;font-weight:700;letter-spacing:.8px">模块 8：视图与物化视图</div><div style="color:#17212b;font-size:30px;font-weight:750">模块 8：视图与物化视图</div><p style="color:#475569;line-height:1.7">把稳定的业务查询封装成逻辑视图，再用物化视图预计算重复聚合。请按顺序运行；结果会以表格展示，写入只作用于本模块的 `_l2` 对象。</p></div>

## 边界

本实验不会修改 Level 1 源表，只创建或替换带 `_l2` 后缀的对象。

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course import WarehouseLab

lab = WarehouseLab()


In [ ]:
lab.execute("DROP VIEW IF EXISTS orders_service_view_l2")
lab.execute("DROP MATERIALIZED VIEW IF EXISTS orders_daily_mv_l2")
lab.execute("""
CREATE VIEW orders_service_view_l2 AS
SELECT DATE(event_time) AS order_date, customer_id, order_amount, data_source
FROM orders_imported
WHERE order_amount > 0
""")
lab.sql("SELECT * FROM orders_service_view_l2 ORDER BY order_date, customer_id LIMIT 10", title="稳定的服务视图")

In [ ]:
lab.execute("""
CREATE MATERIALIZED VIEW orders_daily_mv_l2
BUILD IMMEDIATE
REFRESH AUTO ON MANUAL
AS
SELECT DATE(event_time) AS order_date, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
FROM orders_imported
GROUP BY DATE(event_time)
""")
lab.sql(f"SELECT * FROM mv_infos('database'='{lab.database}') WHERE Name = 'orders_daily_mv_l2'", title="物化视图元数据")

In [ ]:
lab.sql("""
SELECT order_date, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
FROM orders_service_view_l2
GROUP BY order_date
ORDER BY order_date
""", title="规范的日指标")
lab.sql("""
EXPLAIN
SELECT DATE(event_time) AS order_date, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
FROM orders_imported
GROUP BY DATE(event_time)
ORDER BY order_date
""", title="重复指标的查询改写证据")

## 要点

将结果与课程中声明的业务粒度对照。SQL 执行成功本身不能证明模型、指标、访问边界或消费者契约正确。